# CICIoT2023 — CSV-first training and deployment
Only the given `himadri07/ciciot2023` Kaggle CSV dataset is used. No PCAPs or new recordings are required for training.
Train, validation and test files retain their provided assignments. Test labels never determine training cleaning. Capture/session independence remains unknown.
The model learns from 18 original CSV measurements plus four shared engineered features. Raw-PCAP equivalence has not been established; do not relabel an approximate packet extractor as compatible.

In [ ]:
# Run from this repository checkout (upload/clone it into the Kaggle notebook first).
%pip install -e . xgboost==3.4.1


In [ ]:
from pathlib import Path
from sentinel_ml.csv_training import train
from sentinel_ml.csv_contract import SPEC, CONTRACT_HASH
import json
DATASET_ROOT = Path("/kaggle/input/ciciot2023")
OUTPUT = Path("/kaggle/working/ciciot23_artifacts")
print(json.dumps(SPEC, indent=2))
print(CONTRACT_HASH)
assert DATASET_ROOT.exists(), "Set DATASET_ROOT to the mounted himadri07/ciciot2023 dataset"


## Training protocol
Rows outside BenignTraffic/DoS-*/DDoS-* are excluded. Nonfinite base values and negative duration/rate rows are excluded with counts. No imputation or arbitrary physical packet bounds are applied to original CSV statistics. Exact same-label training duplicates are removed using training only; ambiguous training vectors remain. Holdout rows remain intact; overlap is reported.
XGBoost, Random Forest and scaled Logistic Regression use the same 22 ordered float32 inputs. Choose using validation macro-F1. Only the selected model receives final test evaluation. Freeze this experiment before looking at test results; do not tune from the final score.
The supplied row splits do not certify unseen-device/session generalization. Report the caveat and cross-split feature overlap with metrics.

In [ ]:
report = train(DATASET_ROOT, OUTPUT)
print(json.dumps(report, indent=2))


In [ ]:
from sentinel_ml.artifacts import load
for model_id in report["artifact_hashes"]:
    bundle = load(OUTPUT / f"ciciot_22f_{model_id}.joblib")
    print(model_id, bundle["artifact_sha256"], "CSV contract and reference predictions verified")


## Application
Set MODEL_DIR to the exported directory. FastAPI CSV upload uses the same preprocessor and validates golden probabilities on artifact load. Extra Kaggle columns and labels are excluded from inference.
PCAP-to-model support requires an extractor that reproduces the original base statistics, verified on paired original captures and feature rows. Those pairs and the matching generator revision are not in the supplied CSV dataset; no distribution-matching or guessed rescaling is presented as a solution.
